In [12]:
import cv2
import numpy as np
from keras.models import load_model
from collections import deque
from IPython.display import display, clear_output
import time
from PIL import Image
import io

def display_video_fluid(video_path, model_path='modelfinal.h5', limit=None, target_fps=24):
    # Cargar modelo optimizado
    model = load_model(model_path, compile=False)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    Q = deque(maxlen=128)
    cap = cv2.VideoCapture(video_path)
    
    # Configuración de timing
    frame_delay = 1/target_fps
    last_time = time.time()
    
    # Precalentar modelo (para primera inferencia más rápida)
    model.predict(np.zeros((1, 128, 128, 3)))
    
    try:
        while True:
            start_time = time.time()
            ret, frame = cap.read()
            if not ret or (limit and count >= limit):
                break
                
            # Procesamiento eficiente
            frame_small = cv2.resize(frame, (128, 128))
            frame_rgb = cv2.cvtColor(frame_small, cv2.COLOR_BGR2RGB)
            frame_norm = frame_rgb.astype("float32") / 255
            
            # Inferencia batch (más rápida)
            preds = model.predict(np.expand_dims(frame_norm, axis=0), verbose=0)[0]
            Q.append(preds)
            label = (preds > 0.50)[0]
            
            # Dibujar resultado
            text_color = (0, 0, 255) if label else (0, 255, 0)
            cv2.putText(frame, f"Violence: {label}", (35, 50), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, text_color, 2)
            
            # Mostrar frame optimizado
            display(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            clear_output(wait=True)
            
            # Control de FPS preciso
            processing_time = time.time() - start_time
            sleep_time = max(0, frame_delay - processing_time)
            time.sleep(sleep_time)
            
    finally:
        cap.release()
        clear_output()

# Uso con FPS objetivo de 24 (ajustable)

In [14]:
display_video_fluid("V1.mp4", target_fps=24)